In [ ]:
 # === Predict TTS from intergenic epigenetic profiles (±1000bp, 20 bins) ===
import os, math, glob
import numpy as np
import pandas as pd
from functools import reduce
import joblib

# ------------------------
# Paths (edit if needed)
# ------------------------
MODEL_PATH = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/model_training_end/rf_groupcv_out/rf_groupcv_pipeline.joblib"
PRED_DIR   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/intergenic_bins"
OUT_DIR    = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TTS"
os.makedirs(OUT_DIR, exist_ok=True)

# Feature files to include (same set as before)
INCLUDE_FILES = [
    "ATAC_intergenic_bins_pm1000bp_20bins.csv",
    "H2A.Zac_intergenic_bins_pm1000bp_20bins.csv",
    "H2A.Z_intergenic_bins_pm1000bp_20bins.csv",
    "H2B.Z_intergenic_bins_pm1000bp_20bins.csv",
    "H3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K18ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3K18me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K27ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3K27me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K36me3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me1_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me2_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K9ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3R17me2_intergenic_bins_pm1000bp_20bins.csv",
    "MNase_intergenic_bins_pm1000bp_20bins.csv",
]

# ------------------------
# Parameters
# ------------------------
KEY_COLS = ["chr","mid","ID","strand"]
BIN_WIDTH = 100               # 20 x 100bp bins for ±1000bp
THRESHOLD = 0.7               # classify positives at this prob (adjust as you like)
MIN_PROB_FOR_PEAK = 0.7       # require at least this prob to consider a local peak
PEAK_NEIGHBORHOOD = 3         # ±3 bins (±300 bp) for local maxima
MIN_PEAK_DISTANCE_BP = 200    # greedily keep peaks ≥200 bp apart


In [ ]:
# ------------------------
# Helpers
# ------------------------
def load_one(path):
    df = pd.read_csv(path)
    miss = [c for c in KEY_COLS if c not in df.columns]
    if miss:
        raise ValueError(f"{os.path.basename(path)} missing key cols {miss}")
    for c in df.columns:
        if c not in KEY_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def infer_expected_feature_order(pipeline, candidate_cols):
    # try to recover training feature order
    for name, step in getattr(pipeline, "named_steps", {}).items():
        if hasattr(step, "feature_names_in_"):
            return list(step.feature_names_in_)
    if hasattr(pipeline, "feature_names_in_"):
        return list(pipeline.feature_names_in_)
    return list(candidate_cols)

def choose_source(expected, cols):
    """Pick a column from merged that should feed 'expected'.
       Handles '_TTS' naming differences."""
    if expected in cols:
        return expected
    no_tts = expected.replace("_TSS", "").replace("_TTS", "")
    if no_tts in cols:
        return no_tts
    # You can add more normalization rules here if needed
    return None

def local_maxima_1d(a, neighborhood=3):
    n = len(a)
    if n == 0:
        return np.zeros(0, dtype=bool)
    peaks = np.ones(n, dtype=bool)
    for k in range(1, neighborhood+1):
        left  = np.r_[a[k:], np.full(k, -np.inf)]
        right = np.r_[np.full(k, -np.inf), a[:-k]]
        peaks &= (a > left) & (a > right)
    return peaks

def prune_by_distance(df, min_dist_bp=200):
    keep = []
    last_by_cs = {}
    for _, row in df.sort_values("score", ascending=False).iterrows():
        key = (row["chr"], row["strand"])
        last = last_by_cs.get(key, None)
        if last is None or abs(int(row["mid"]) - int(last)) >= min_dist_bp:
            keep.append(row)
            last_by_cs[key] = int(row["mid"])
    return pd.DataFrame(keep)

In [ ]:
# ------------------------
# 1) Read & merge feature tables
# ------------------------
frames = []
for fname in INCLUDE_FILES:
    fpath = os.path.join(PRED_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[WARN] missing -> {fname}")
        continue
    print("Reading", fname, "...")
    frames.append(load_one(fpath))

if not frames:
    raise RuntimeError("No input files found.")

print("Merging feature tables ...")
merged = reduce(lambda l, r: pd.merge(l, r, on=KEY_COLS, how="outer"), frames)
# Fill missing with 0 for signal columns
for c in merged.columns:
    if c not in KEY_COLS:
        merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0.0)

print(merged.shape)
print(merged.head())

In [ ]:
# ------------------------
# 2) Load TTS model & build X
# ------------------------
pipe = joblib.load(MODEL_PATH)
print(pipe)

numeric_cols = [c for c in merged.columns if c not in KEY_COLS]
expected_cols = infer_expected_feature_order(pipe, numeric_cols)
print("Expected features:", len(expected_cols))

# Build X fast (vectorized)
cols_set = set(numeric_cols)
series_list, names = [], []
for col in expected_cols:
    src = choose_source(col, cols_set)
    if src is None:
        s = pd.Series(0.0, index=merged.index)
    else:
        s = pd.to_numeric(merged[src], errors="coerce").fillna(0.0)
    series_list.append(s.astype("float32"))
    names.append(col)

X = pd.concat(series_list, axis=1)
X.columns = names
X = X.copy()  # de-fragment

print("NaNs in X:", int(X.isna().sum().sum()))
print("Nonzero entries in X:", int((X != 0).sum().sum()))
print("X shape:", X.shape)

In [ ]:
# ------------------------
# 3) Predict probabilities
# ------------------------
if hasattr(pipe, "predict_proba"):
    proba = pipe.predict_proba(X)
    classes_ = pipe.classes_ if hasattr(pipe, "classes_") else np.array([0,1])
    pos_idx = list(classes_).index(1) if 1 in classes_ else (1 if proba.shape[1] > 1 else 0)
    scores = proba[:, pos_idx].astype("float32")
else:
    scores = (1/(1+np.exp(-pipe.decision_function(X).astype(float)))).astype("float32") \
             if hasattr(pipe, "decision_function") else pipe.predict(X).astype("float32")

preds = (scores >= THRESHOLD).astype("int8")
print(pd.Series(scores).describe())
print("# positives @ threshold", THRESHOLD, "->", int(preds.sum()))


In [ ]:
# ------------------------
# 4) Save per-site predictions
# ------------------------
keys_df = merged[KEY_COLS].reset_index(drop=True)
pred_df = keys_df.copy()
pred_df["score"] = scores
pred_df["pred_label"] = preds
out_pred_csv = os.path.join(OUT_DIR, f"predictions_all_TTS_thr{THRESHOLD}.csv")
pred_df.to_csv(out_pred_csv, index=False)
print("Saved:", out_pred_csv)

In [ ]:
# ------------------------
# 5) Continuous track: bedGraph
# ------------------------
start = (pred_df["mid"].astype(int) - (BIN_WIDTH // 2)).clip(lower=0)
end   = start + BIN_WIDTH
bedgraph = pd.DataFrame({
    "chr":   pred_df["chr"].astype(str),
    "start": start.astype(int),
    "end":   end.astype(int),
    "score": pred_df["score"].astype(float),
})
chrom_ordered = bedgraph.sort_values(["chr","start","end"], kind="mergesort").reset_index(drop=True)
bg_path = os.path.join(OUT_DIR, f"predictions_TTS_thr{THRESHOLD}.bedgraph")
chrom_ordered.to_csv(bg_path, sep="\t", header=False, index=False)
print("Saved:", bg_path)


In [ ]:
# ------------------------
# 6) Local-peak calling for TTS candidates
# ------------------------
cand_list = []
for (chrom, strand), grp in pred_df.groupby(["chr","strand"], sort=False):
    a = grp["score"].to_numpy()
    peaks_mask = local_maxima_1d(a, neighborhood=PEAK_NEIGHBORHOOD)
    peaks_df = grp.loc[peaks_mask].copy()
    peaks_df = peaks_df[peaks_df["score"] >= MIN_PROB_FOR_PEAK]
    cand_list.append(peaks_df)

cands = pd.concat(cand_list, ignore_index=True) if cand_list else pred_df.iloc[0:0].copy()
print("Raw local peaks:", len(cands))

cands_pruned = prune_by_distance(cands, min_dist_bp=MIN_PEAK_DISTANCE_BP)
print("Peaks after distance pruning:", len(cands_pruned))


In [ ]:
# BED of TTS candidates
tts_bed = pd.DataFrame({
    "chr":    cands_pruned["chr"].astype(str),
    "start": (cands_pruned["mid"].astype(int) - (BIN_WIDTH//2)).clip(lower=0),
    "end":   (cands_pruned["mid"].astype(int) + (BIN_WIDTH//2)),
    "name":  cands_pruned["ID"].astype(str),
    "score": (cands_pruned["score"]*1000).round().astype(int),
    "strand": cands_pruned["strand"].astype(str),
}).sort_values(["chr","start","end"], kind="mergesort")
tts_bed_path = os.path.join(OUT_DIR, f"tts_candidates_thr{MIN_PROB_FOR_PEAK}_nb{PEAK_NEIGHBORHOOD}_d{MIN_PEAK_DISTANCE_BP}.bed")
tts_bed.to_csv(tts_bed_path, sep="\t", header=False, index=False)
print("Saved:", tts_bed_path)